In [0]:
%pip install python-dotenv

In [0]:
%pip install azure-storage-file-datalake azure-identity python-dotenv

## 1. Carregar variáveis de ambiente



In [0]:

from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv("../.env")

CLIENT_ID = os.getenv("client_id")
TENANT_ID = os.getenv("tenant_id")
CLIENT_SECRET = os.getenv("client_secret")
STORAGE_ACCOUNT_NAME = os.getenv("storage_account_name")
CONTAINER_NAME = os.getenv("container_name", "raw")
RAW_FOLDER = os.getenv("raw_folder", "real-time-data")

## 2. Conectar ao ADLS Gen2

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

account_url = f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"

service_client = DataLakeServiceClient(
    account_url=account_url,
    credential=credential
)

file_system_client = service_client.get_file_system_client(CONTAINER_NAME)

## 3. Localizar arquivos `ecommerce_enderecos.parquet` na pasta raw


In [0]:
for fs in service_client.list_file_systems():
    print(fs.name)

## 4. Ler todos os Parquets e unir em um único DataFrame


In [0]:
arquivos_enderecos = [
    p.name
    for p in file_system_client.get_paths()
    if p.name.endswith("ecommerce_enderecos.parquet")
]

print(f"Arquivos encontrados: {len(arquivos_enderecos)}")

for arquivo in arquivos_enderecos:
    print(arquivo)

BUSCANDO OS ARQUIVOS E FAZENDO A JUNÇÃO DELES

In [0]:
import os

destino = "/tmp/ecommerce_enderecos"
os.makedirs(destino, exist_ok=True)

for i, arquivo in enumerate(arquivos_enderecos):
    file_client = file_system_client.get_file_client(arquivo)

    local_path = f"{destino}/ecommerce_enderecos_{i}.parquet"

    with open(local_path, "wb") as f:
        f.write(file_client.download_file().readall())

In [0]:
import os

                                                                                                                                          
print(os.listdir("/tmp/ecommerce_enderecos"))

TAMANHO DOS ARQUIVOS


In [0]:
arquivos_info = []

for arquivo in arquivos_enderecos:
    props = file_system_client.get_file_client(arquivo).get_file_properties()
    
    arquivos_info.append({
        "arquivo": arquivo,
        "tamanho_bytes": props.size,
        "tamanho_mb": round(props.size / 1024 / 1024, 2)
    })

for item in arquivos_info:
    print(item["tamanho_mb"], "MB -", item["arquivo"])

In [0]:
total_bytes = sum(item["tamanho_bytes"] for item in arquivos_info)
total_mb = total_bytes / 1024 / 1024
total_gb = total_mb / 1024

print(f"Total: {total_mb:.2f} MB")
print(f"Total: {total_gb:.2f} GB")

In [0]:
if total_mb < 500:
    print("Pode usar Pandas com BytesIO tranquilamente.")
elif total_mb < 2000:
    print("Dá para tentar Pandas, mas com cuidado.")
else:
    print("Melhor não usar Pandas. Precisaremos pensar em outra estratégia para Serverless.")

CONECTANDO AO BANCO DE DADOS


In [0]:
df_teste = spark.createDataFrame(arquivos_info)

df_teste.printSchema()
print("Total de registros:", df_teste.count())

display(df_teste.limit(5))


In [0]:
JDBC_HOSTNAME = os.getenv("jdbc_hostname")
JDBC_DATABASE = os.getenv("jdbc_database")
JDBC_USERNAME = os.getenv("jdbc_username")
JDBC_PASSWORD = os.getenv("jdbc_password")

In [0]:
jdbc_url = (
    f"jdbc:sqlserver://{JDBC_HOSTNAME}:1433;"
    f"database={JDBC_DATABASE};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user": JDBC_USERNAME,
    "password": JDBC_PASSWORD,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
%skip
(
    df_teste.write
    .format("sqlserver")
    .mode("overwrite")
    .option("host", JDBC_HOSTNAME)
    .option("port", "1433")
    .option("database", JDBC_DATABASE)
    .option("user", JDBC_USERNAME)
    .option("password", JDBC_PASSWORD)
    .option("dbtable", "squad1.ecommerce_enderecos")
    .option("encrypt", "true")
    .option("trustServerCertificate", "false")
    .save()
)